[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/04_NumPy.ipynb)

# NumPy Arrays

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 3

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- explain why physicists use NumPy arrays instead of Python lists for numerical work;
- create arrays with `np.array`, `np.zeros`, `np.ones`, `np.arange`, and `np.linspace`, and check their `dtype` and `shape`;
- do arithmetic on whole arrays at once ("vectorization") and use NumPy's math functions (ufuncs);
- measure how much faster vectorized code is than a Python loop;
- pick out pieces of an array with indexing, slicing, and boolean masks, and use `np.where`;
- summarize data with `sum`, `mean`, `std`, `min`, `max`, and `argmax`, including along one axis of a 2D array;
- reshape arrays and understand the basic rules of broadcasting;
- read a data file with `np.loadtxt` and write one with `np.savetxt`.

Physics along the way: relativistic $\gamma$ for many speeds at once, the Planck blackbody spectrum and Wien's law, and particle energies.

## 1. Why arrays?

In notebook 02 you stored many numbers in a Python **list** and used a `for` loop to do something to each one. That works, but it has two problems for physics:

1. **It is clumsy.** Lists do not know how to do math. Suppose you have a list of speeds and want to double each one:

In [ ]:
speeds = [1.0, 2.0, 3.0]
print(speeds * 2)       # not what you wanted!
print(speeds + speeds)  # also not math: this glues the lists together

For a list, `*` means "repeat" and `+` means "join". To actually double every element you need a loop or a list comprehension:

In [ ]:
doubled = []
for v in speeds:
    doubled.append(2 * v)
print(doubled)

2. **It is slow.** Python checks the type of every single element every time through the loop. For a few numbers you will never notice, but simulations and data sets often have millions of numbers.

**NumPy** ("Numerical Python") solves both problems. Its central object is the **array**: a grid of numbers that all have the *same type*, stored next to each other in memory. Arithmetic on an array acts on every element at once, and the actual loop runs in fast compiled C code.

By convention everyone imports NumPy as `np`:

In [ ]:
import numpy as np

speeds = np.array([1.0, 2.0, 3.0])
print(speeds * 2)
print(speeds + speeds)

Now `*` and `+` do what a physicist expects. Notice that NumPy prints arrays without commas, which is a quick way to tell an array from a list.

## 2. Creating arrays

### 2.1 From a list: `np.array`

`np.array(some_list)` converts a list into an array.

In [ ]:
masses_MeV = np.array([0.511, 105.7, 938.3, 939.6])   # e, mu, p, n rest energies in MeV
print(masses_MeV)
print(type(masses_MeV))

### 2.2 `dtype` and `shape`

Every array has a **data type** (`dtype`) shared by all its elements, and a **shape** that says how many elements it has along each direction. These are *attributes*: you access them with a dot and no parentheses.

In [ ]:
print("dtype:", masses_MeV.dtype)
print("shape:", masses_MeV.shape)
print("ndim: ", masses_MeV.ndim)    # number of dimensions
print("size: ", masses_MeV.size)    # total number of elements
print("len:  ", len(masses_MeV))

`float64` means a 64-bit floating-point number, the same as a Python `float`. (Notebook 06 explains exactly what that means.) The shape `(4,)` is a *tuple* with one entry: this is a 1D array of length 4.

If you give NumPy only integers you get an integer array. Mixing integers and floats gives floats, because NumPy picks one type that can hold everything:

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3.5])
print(a, a.dtype)
print(b, b.dtype)

**Pitfall:** an integer array stays an integer array. If you assign a float into it, the value is silently chopped:

In [ ]:
a[0] = 7.9
print(a)    # 7.9 became 7!

For physics you almost always want floats. You can ask for them explicitly with `dtype=float`:

In [ ]:
a = np.array([1, 2, 3], dtype=float)
a[0] = 7.9
print(a, a.dtype)

### 2.3 Filled arrays: `np.zeros` and `np.ones`

Often you need an array of a certain size that you will fill in later (for example, positions at each time step of a simulation).

In [ ]:
z = np.zeros(5)
o = np.ones(3)
print(z)
print(o)
print(np.full(4, 9.81))   # every element equal to 9.81

### 2.4 Ranges: `np.arange` and `np.linspace`

`np.arange(start, stop, step)` works like `range`, but returns an array and allows non-integer steps. As with `range`, the `stop` value is **not** included.

`np.linspace(start, stop, num)` gives `num` evenly spaced points, and **does** include both ends. For physics grids (e.g. 500 wavelengths between 100 nm and 3000 nm) `linspace` is usually what you want, because you control exactly how many points you get.

In [ ]:
print(np.arange(0, 10, 2))       # 0, 2, 4, 6, 8   (10 not included)
print(np.arange(0, 1, 0.25))
print(np.linspace(0, 1, 5))      # 5 points, both ends included

**Pitfall:** with a non-integer step, round-off error can make `arange` include or skip the last point unexpectedly (you will see why in notebook 06). Prefer `linspace` when the step is not an integer.

There is also `np.logspace(a, b, n)`, which gives `n` points from $10^a$ to $10^b$, evenly spaced on a log scale. It is handy when a quantity spans many orders of magnitude.

In [ ]:
print(np.logspace(-3, 0, 4))    # 0.001, 0.01, 0.1, 1

### Your turn 1

Create (a) an array of the integers 1 through 10, (b) 7 evenly spaced times from 0 to 3 s, and (c) an array of 6 zeros. Print each one together with its `dtype` and `shape`.

In [ ]:
# Your code here

## 3. Vectorized arithmetic and ufuncs

### 3.1 Element-by-element arithmetic

Arithmetic between an array and a number, or between two arrays of the same shape, happens **element by element**. Writing code this way, without explicit loops, is called **vectorization**.

In [ ]:
t = np.linspace(0, 3, 7)          # times in s
g = 9.81                          # m/s^2
y = 20.0 - 0.5 * g * t**2         # height of a ball dropped from 20 m
print("t =", t)
print("y =", y)

In [ ]:
x = np.array([1.0, 2.0, 3.0])
w = np.array([10.0, 20.0, 30.0])
print(x + w)
print(x * w)     # element-by-element product, NOT a dot product
print(w / x)
print(x ** 2)

If you do want the **dot product** $\sum_i x_i w_i$ (multiply matching elements, then add them up; you have met it as $\vec a\cdot\vec b = a_x b_x + a_y b_y + a_z b_z$), use `np.dot(x, w)` or the `@` operator:

In [ ]:
print(np.dot(x, w), x @ w)

### 3.2 Universal functions (ufuncs)

The `math` module's functions (`math.sqrt`, `math.exp`, ...) only accept a single number. NumPy has its own versions that accept whole arrays and apply the function to every element. These are called **universal functions** or **ufuncs**: `np.sqrt`, `np.exp`, `np.log`, `np.log10`, `np.sin`, `np.cos`, `np.abs`, and many more.

In [ ]:
import math

angles = np.linspace(0, np.pi, 5)
print(np.sin(angles))
print(np.exp(np.array([0.0, 1.0, 2.0])))

try:
    math.sin(angles)
except TypeError as err:
    print("math.sin fails on an array:", err)

Rule of thumb: once you have imported NumPy, use `np.sqrt`, `np.exp`, etc. everywhere. They work on single numbers too. NumPy also provides `np.pi` and `np.e`.

### 3.3 Example: Lorentz factor for many speeds at once

Quick physics recap: in special relativity, an object moving at speed $v$ has its clocks run slow and its lengths contracted by the **Lorentz factor**

$$\gamma = \frac{1}{\sqrt{1-\beta^2}}, \qquad \beta = v/c,$$

where $c$ is the speed of light. $\gamma = 1$ at rest and grows without limit as $v \to c$. In notebook 02 you would have looped over speeds. With NumPy it is one line:

In [ ]:
beta = np.array([0.0, 0.1, 0.5, 0.9, 0.99, 0.999])
gamma = 1 / np.sqrt(1 - beta**2)
for b, gm in zip(beta, gamma):
    print(f"beta = {b:6.3f}   gamma = {gm:8.3f}")

(`zip(a, b)` walks through two sequences side by side; it is just a convenient way to print a table here. The computation itself had no loop.)

A function you write with NumPy operations automatically works on arrays too:

In [ ]:
from scipy import constants as const

def lorentz_gamma(v):
    """Lorentz factor for speed v in m/s (number or array)."""
    beta = v / const.c
    return 1 / np.sqrt(1 - beta**2)

print(lorentz_gamma(3.0e7))                          # one number
print(lorentz_gamma(np.array([3e7, 1.5e8, 2.9e8])))  # an array

### 3.4 Example: energies of a particle

For a particle of mass $m$ with speed $v$, special relativity says the total energy is $E = \gamma m c^2$. At rest this is the rest energy $mc^2$, so the energy of motion (kinetic energy) is $K = E - mc^2 = (\gamma - 1) m c^2$. For $v \ll c$ this reduces to the familiar $\tfrac12 mv^2$. Let's compute the kinetic energy of an electron for 6 speeds, in MeV.

In [ ]:
m_e = const.m_e                  # kg
MeV = 1e6 * const.e              # 1 MeV in joules
beta = np.linspace(0.1, 0.9, 6)
gamma = 1 / np.sqrt(1 - beta**2)

K_rel = (gamma - 1) * m_e * const.c**2 / MeV
K_classical = 0.5 * m_e * (beta * const.c)**2 / MeV
print("beta       :", beta)
print("K_rel (MeV):", K_rel)
print("K_cl  (MeV):", K_classical)
print("ratio      :", K_rel / K_classical)

At $\beta = 0.1$ the classical formula is within about 1%, but at $\beta = 0.9$ it is off by a factor of several. You will study this carefully in Project 1.

You can also control how arrays print, which helps when there are many digits. `np.set_printoptions(precision=3)` shows 3 decimal places from then on:

In [ ]:
np.set_printoptions(precision=4)
print(K_rel / K_classical)

### Your turn 2

A photon of wavelength $\lambda$ has energy $E = hc/\lambda$. Make an array of the wavelengths 400, 450, 500, 550, 600, 650, 700 nm, and compute the photon energies **in eV** in one vectorized line. Use `scipy.constants` for $h$, $c$, and $e$. Which wavelength gives the most energetic photon?

In [ ]:
# Your code here

## 4. How much faster is vectorized code?

Let's time the two approaches on 100,000 speeds. The `time` module's `time.perf_counter()` returns a clock reading in seconds; the difference between two readings is the elapsed time.

In [ ]:
import time

N = 100_000                       # underscores in numbers are allowed, for readability
beta_list = [i / N for i in range(N)]
beta_arr = np.array(beta_list)

t0 = time.perf_counter()
gamma_list = []
for b in beta_list:
    gamma_list.append(1 / math.sqrt(1 - b**2))
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
gamma_arr = 1 / np.sqrt(1 - beta_arr**2)
t_vec = time.perf_counter() - t0

print(f"loop:       {t_loop*1e3:8.3f} ms")
print(f"vectorized: {t_vec*1e3:8.3f} ms")
print(f"speed-up:   {t_loop/t_vec:8.1f} x")

The exact numbers depend on your computer, but the vectorized version is typically tens to hundreds of times faster. For a simulation that would take an hour with loops, that is the difference between an hour and a minute.

Jupyter also has a built-in **magic command** for timing, `%timeit`, that runs a line many times and reports the average. Magic commands start with `%` and only work in notebooks, not in ordinary Python scripts.

In [ ]:
%timeit -n 10 -r 3 1 / np.sqrt(1 - beta_arr**2)

(`-n 10 -r 3` means "run 10 times per repeat, 3 repeats". Without these options `%timeit` decides for itself, which can take a while for slow code.)

Before trusting a speed-up, check that both methods give the same answer:

In [ ]:
print(np.allclose(gamma_list, gamma_arr))

`np.allclose` returns `True` if all elements agree to within a tiny tolerance. (Why not use `==`? See notebook 06.)

## 5. Indexing and slicing

Indexing and slicing work just like they do for lists: indices start at 0, negative indices count from the end, and `a[start:stop:step]` excludes `stop`.

In [ ]:
a = np.arange(10, 20)
print(a)
print(a[0], a[3], a[-1])
print(a[2:5])      # elements 2, 3, 4
print(a[:3])       # first three
print(a[-3:])      # last three
print(a[::2])      # every other element
print(a[::-1])     # reversed

You can assign to an element or a whole slice:

In [ ]:
a[0] = -1
a[5:8] = 0
print(a)

**Pitfall: slices are views, not copies.** A slice of an array shares memory with the original. Changing the slice changes the original! If you want an independent copy, use `.copy()`.

In [ ]:
a = np.arange(5)
s = a[1:3]
s[0] = 100
print("a after changing the slice:", a)

a = np.arange(5)
s = a[1:3].copy()
s[0] = 100
print("a after changing a copy:   ", a)

A very common use of slicing in physics is computing differences between neighbouring elements, e.g. velocity from positions: $v_i \approx (x_{i+1} - x_i)/\Delta t$. `x[1:]` is "all but the first" and `x[:-1]` is "all but the last", so their difference is the array of neighbour differences. (NumPy also has `np.diff` for this.)

In [ ]:
dt = 0.5
t = np.arange(0, 3.01, dt)
x = 20.0 - 0.5 * 9.81 * t**2
v = (x[1:] - x[:-1]) / dt
print(v)
print(np.diff(x) / dt)

## 6. Boolean masks and `np.where`

Comparing an array to something gives an array of `True`/`False` values, one per element:

In [ ]:
beta = np.array([0.05, 0.2, 0.5, 0.8, 0.95, 0.99])
gamma = 1 / np.sqrt(1 - beta**2)
print(gamma)
print(gamma > 2)

Such an array is called a **boolean mask**. If you index an array with a mask, you get only the elements where the mask is `True`. This is how you select data that pass a "cut", which is exactly what particle physicists do all day.

In [ ]:
fast = gamma > 2
print("speeds with gamma > 2:", beta[fast])
print("how many:", np.sum(fast))      # True counts as 1, False as 0

To combine conditions use `&` (and), `|` (or), and `~` (not), with **parentheses around each condition**. The Python words `and`/`or` do not work on arrays.

In [ ]:
mid = (beta > 0.1) & (beta < 0.9)
print(beta[mid])
print(beta[~mid])

try:
    beta[(beta > 0.1) and (beta < 0.9)]
except ValueError as err:
    print("Using 'and' gives an error:", err)

That error message ("The truth value of an array with more than one element is ambiguous") is one you will meet often. It almost always means you used `and`, `or`, or `if` on an array instead of a single `True`/`False`.

**`np.where(condition, x, y)`** builds a new array, taking elements from `x` where the condition is true and from `y` where it is false. For example, "classical" vs "relativistic" labels, or clipping negative values:

In [ ]:
K_rel = (gamma - 1)                  # kinetic energy in units of mc^2
K_cl = 0.5 * beta**2
K_best_cheap = np.where(beta < 0.1, K_cl, K_rel)   # use the classical formula only when it is accurate
print(K_best_cheap)

signal = np.array([3.0, -0.5, 2.0, -1.2, 0.4])
print(np.where(signal > 0, signal, 0.0))           # set negative readings to zero

With only a condition, `np.where(condition)` returns the **indices** where it is true:

In [ ]:
print(np.where(gamma > 2))
print(np.where(gamma > 2)[0])     # the indices themselves are the first element

### Your turn 3

The array below holds simulated (made-up) measured energies of particles in GeV. (a) How many particles have energy above 50 GeV? (b) What is the average energy of the particles between 10 and 50 GeV? (c) Use `np.where` to make a new array in which any energy above 100 GeV is replaced by 100 (a detector "saturating").

In [ ]:
rng = np.random.default_rng(42)
E_GeV = rng.exponential(30.0, size=1000)   # simulated energies, GeV
# Your code here

(`rng.exponential` draws random numbers from an exponential distribution; you will learn about random numbers properly in notebook 12. The fixed seed `42` makes everyone get the same numbers.)

## 7. Aggregate functions

Functions that reduce an array to one number are called **aggregates**. They exist both as `np.` functions and as array methods:

In [ ]:
data = np.array([4.2, 3.9, 4.5, 4.1, 3.8, 4.4])
print("sum :", np.sum(data), data.sum())
print("mean:", np.mean(data))
print("std :", np.std(data))
print("min, max:", data.min(), data.max())
print("argmin, argmax:", data.argmin(), data.argmax())

`argmax` returns the **index** of the largest element, not the value. That is useful when you want to know *where* something peaks, as in the next example.

A note on `std`: by default `np.std` divides by $N$. The usual "sample standard deviation" from a lab course divides by $N-1$; ask for it with `np.std(data, ddof=1)`. (`ddof` means "delta degrees of freedom".)

In [ ]:
print(np.std(data), np.std(data, ddof=1))

Other handy ones: `np.median`, `np.cumsum` (running total), `np.prod`, `np.any`, `np.all`.

### 7.1 Physics example: the blackbody spectrum and Wien's law

Physics recap: every hot object glows. An idealized perfect absorber and emitter, a **blackbody**, emits a spectrum that depends only on its temperature. Hotter objects are brighter and their spectrum peaks at shorter wavelength (a stove element glows red, the Sun looks white-yellow). Planck explained the shape in 1900 by assuming light is emitted in quanta of energy $hf$, which launched quantum mechanics. The brightness per unit wavelength (spectral radiance) is Planck's law:

$$
B_\lambda(\lambda, T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{e^{hc/(\lambda k_B T)} - 1}
\qquad [\mathrm{W\,sr^{-1}\,m^{-3}}].
$$

Let's evaluate it on a fine wavelength grid with no loops.

In [ ]:
h, c, kB = const.h, const.c, const.k

def planck(lam, T):
    """Planck spectral radiance B_lambda in W / (sr m^3); lam in m, T in K."""
    return 2 * h * c**2 / lam**5 / np.expm1(h * c / (lam * kB * T))

lam = np.linspace(100e-9, 3000e-9, 5000)    # 100 nm to 3000 nm
B = planck(lam, 5800.0)                     # roughly the Sun's surface temperature
print(B.shape, B.max())

`np.expm1(x)` computes $e^x - 1$. It is mathematically the same as `np.exp(x) - 1` but more accurate when $x$ is small; notebook 06 explains why.

Now find the peak with `argmax` and compare with **Wien's displacement law**, $\lambda_\text{max} T = b$ with $b \approx 2.898\times10^{-3}$ m K. SciPy has $b$ as `const.Wien`.

In [ ]:
i_peak = np.argmax(B)
lam_peak = lam[i_peak]
print(f"peak index: {i_peak}")
print(f"peak wavelength: {lam_peak*1e9:.1f} nm")
print(f"lambda_max * T = {lam_peak * 5800:.4e} m K")
print(f"Wien's constant = {const.Wien:.4e} m K")

They agree to about the grid spacing. The Sun's spectrum peaks near 500 nm, in the green part of the visible.

A quick picture helps. You will learn plotting properly in the next notebook; for now, `plt.plot(x, y)` is enough.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(lam * 1e9, B)
plt.axvline(lam_peak * 1e9, color="gray", linestyle="--")
plt.xlabel("wavelength (nm)")
plt.ylabel("spectral radiance (W sr$^{-1}$ m$^{-3}$)")
plt.show()

### Your turn 4

Repeat the Wien's-law check for $T$ = 3000 K, 4000 K, 6000 K, and 10000 K. For each temperature find $\lambda_\text{max}$ with `argmax` and print $\lambda_\text{max} T$. (A `for` loop over the four temperatures is fine; the work inside it should be vectorized over wavelength.) Make sure your wavelength grid is wide enough to contain the peak for every temperature.

In [ ]:
# Your code here

## 8. Two-dimensional arrays and the `axis` argument

Arrays can have more than one dimension. A 2D array is like a table of numbers with rows and columns (mathematicians call such a table a *matrix*): you make one from a list of lists (each inner list is a row).

In [ ]:
M = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
print(M)
print("shape:", M.shape)     # (rows, columns)

Index with `M[row, column]`. A colon `:` on its own means "everything along this direction".

In [ ]:
print(M[0, 2])      # row 0, column 2
print(M[1, :])      # all of row 1
print(M[:, 0])      # all of column 0
print(M[:, 1:])     # columns 1 onward

`np.zeros` and `np.ones` take a shape tuple for 2D arrays:

In [ ]:
grid = np.zeros((3, 4))
print(grid)

### 8.1 The `axis` argument

Aggregate functions on a 2D array act on *all* elements by default. With `axis=0` they work **down the rows** (one result per column); with `axis=1` they work **across the columns** (one result per row). A way to remember it: `axis=0` is the direction that gets collapsed.

Example: three students each measured $g$ (in m/s²) in four trials. Rows are students, columns are trials.

In [ ]:
g_meas = np.array([[9.75, 9.82, 9.79, 9.86],
                   [9.91, 9.70, 9.88, 9.80],
                   [9.81, 9.83, 9.80, 9.82]])

print("overall mean     :", g_meas.mean())
print("mean per student :", g_meas.mean(axis=1))   # average across each row
print("mean per trial   :", g_meas.mean(axis=0))   # average down each column
print("spread per student:", g_meas.std(axis=1, ddof=1))

The third student has the smallest spread, so the most consistent technique.

## 9. Reshaping

`reshape` rearranges the same elements into a different shape. The total number of elements must stay the same.

In [ ]:
a = np.arange(12)
print(a)
print(a.reshape(3, 4))
print(a.reshape(4, 3))
print(a.reshape(2, -1))    # -1 means "work this dimension out for me"

In [ ]:
try:
    a.reshape(5, 2)
except ValueError as err:
    print("Error:", err)

`.flatten()` (or `.ravel()`) goes the other way, turning any array into 1D. `.T` gives the transpose of a 2D array.

In [ ]:
b = a.reshape(3, 4)
print(b.T)
print(b.T.shape)
print(b.flatten())

## 10. Broadcasting

You have already used broadcasting without noticing: in `2 * x`, the single number `2` is "stretched" to match the shape of `x`. NumPy extends this idea to arrays of different shapes. The basic rule is:

> Compare the shapes from the right. Two dimensions are compatible if they are equal or if one of them is 1. A dimension of size 1 is stretched to match the other.

The most useful case for us: a **column** of shape `(n, 1)` combined with a **row** of shape `(m,)` gives an `(n, m)` table of every combination. You can turn a 1D array into a column with `[:, np.newaxis]` (or `.reshape(-1, 1)`).

In [ ]:
row = np.array([10, 20, 30])          # shape (3,)
col = np.array([1, 2])[:, np.newaxis] # shape (2, 1)
print(col.shape, row.shape)
print(col + row)                      # shape (2, 3)

**Physics example:** the Planck spectrum for several temperatures at once. Put the temperatures in a column and the wavelengths in a row; the result has one row per temperature.

In [ ]:
T = np.array([3000.0, 4000.0, 5000.0, 6000.0])[:, np.newaxis]   # shape (4, 1)
lam = np.linspace(100e-9, 5000e-9, 5000)                         # shape (5000,)
B_all = planck(lam, T)                                           # shape (4, 5000)
print(B_all.shape)

i_peaks = np.argmax(B_all, axis=1)       # peak index in each row
print("peak wavelengths (nm):", lam[i_peaks] * 1e9)
print("lambda_max * T (1e-3 m K):", lam[i_peaks] * T[:, 0] * 1e3)

One function call, no loops, and Wien's law holds for every temperature. Notice `planck` did not need to change at all: because it is written with NumPy operations, broadcasting just works.

If the shapes are not compatible you get an error that tells you both shapes. Read it carefully: it usually means you forgot to make one array a column.

In [ ]:
try:
    np.ones(3) + np.ones(4)
except ValueError as err:
    print("Error:", err)

### Your turn 5

In the Bohr model, the electron in hydrogen can only have energies $E_n = -13.6\ \text{eV}/n^2$ ($n = 1, 2, 3, \dots$). When it drops from an upper level to a lower one it emits a photon carrying away the energy difference. Using broadcasting (not loops), build a 2D array whose entry in row $i$, column $j$ is the photon energy $E_{n_\text{upper}} - E_{n_\text{lower}}$ for $n_\text{upper} = 2,\dots,7$ (rows) and $n_\text{lower} = 1, 2, 3$ (columns). Then convert it to wavelengths in nm with $\lambda = hc/E$. Which entries are meaningless (upper level not above lower level), and how could you hide them with `np.where` (for example by setting them to `np.nan`)?

In [ ]:
# Your code here

## 11. Reading and writing data files

### 11.1 Loading with `np.loadtxt`

Most experimental data come as plain text: columns of numbers separated by spaces, tabs, or commas. `np.loadtxt` reads such a file into a 2D array (one row per line).

The course data files live in the `Data/` folder of the course repository. The helper cell below defines a function `data_path(filename)` that finds the file locally if it exists, and otherwise uses its address on GitHub (NumPy can read directly from a web address). Run it once.

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

`millikan.txt` contains Robert Millikan's photoelectric-effect measurements: column 0 is the light frequency (Hz), column 1 is the stopping voltage (V).

In [ ]:
millikan = np.loadtxt(data_path("millikan.txt"))
print(millikan.shape)
print(millikan)

Usually you want each column in its own named array. You can slice the columns out, or use `unpack=True`, which hands back the columns separately:

In [ ]:
freq = millikan[:, 0]
V_stop = millikan[:, 1]

freq, V_stop = np.loadtxt(data_path("millikan.txt"), unpack=True)
print("frequencies (Hz):", freq)
print("stopping V (V)  :", V_stop)

Physics recap: when light shines on a metal it can knock out electrons. Einstein explained this with photons: each photon carries energy $hf$, gives it to one electron, and the electron needs at least the metal's *work function* $\phi$ to escape. The leftover kinetic energy is measured by the reverse voltage $V_\text{stop}$ needed to stop the electrons. So $eV_\text{stop} = hf - \phi$: a straight line in $f$ whose slope of $V_\text{stop}$ vs $f$ is $h/e$. Even with only what you know so far you can get a rough estimate from the first and last points (in the next notebooks you will do this properly with a fit):

In [ ]:
slope = (V_stop[-1] - V_stop[0]) / (freq[-1] - freq[0])
print(f"rough h/e = {slope:.3e} V s")
print(f"true  h/e = {const.h / const.e:.3e} V s")

Useful `np.loadtxt` options (see `help(np.loadtxt)`):

- `delimiter=","` for comma-separated files;
- `skiprows=n` to skip header lines at the top;
- `usecols=(0, 2)` to read only some columns;
- `comments="#"` lines starting with this character are ignored (this is the default).

If a file has missing values or text mixed in, `np.loadtxt` will fail; `np.genfromtxt` is more forgiving and you will use it in the next notebook.

### 11.2 Another data set: velocities

`velocities.txt` holds a velocity measured once per second: column 0 is time (s), column 1 is velocity (m/s).

In [ ]:
t, v = np.loadtxt(data_path("velocities.txt"), unpack=True)
print("number of points:", t.size)
print("time range:", t.min(), "to", t.max(), "s")
print(f"max velocity {v.max():.3f} m/s at t = {t[np.argmax(v)]} s")
print(f"mean velocity {v.mean():.3f} m/s")

### Your turn 6

Using the `t` and `v` arrays from `velocities.txt`:

(a) How many seconds is the velocity negative? (Use a mask.)

(b) Estimate the net displacement $\int v\,dt$ by adding up $v_i\,\Delta t$ with $\Delta t = 1$ s. Then estimate the total distance travelled the same way, but adding up $|v_i|\,\Delta t$ instead (`np.abs`). Why are the two numbers different?

(c) Estimate the acceleration at each time from neighbouring differences (Section 5) and find the largest magnitude.

In [ ]:
# Your code here

### 11.3 Saving with `np.savetxt`

`np.savetxt(filename, array)` writes an array to a text file. To save several 1D arrays as columns, stack them side by side with `np.column_stack`. The `header` text is written at the top, after a `#`, and `fmt` controls the number format.

Here we write to a temporary folder so we do not clutter your files (`tempfile.gettempdir()` gives the computer's temporary directory).

In [ ]:
import tempfile

lam_nm = np.linspace(200, 2000, 10)
B_sun = planck(lam_nm * 1e-9, 5800.0)
table = np.column_stack([lam_nm, B_sun])
print(table.shape)

outfile = os.path.join(tempfile.gettempdir(), "planck_5800K.txt")
np.savetxt(outfile, table, header="wavelength_nm  B_lambda_W_per_sr_m3", fmt="%.6e")
print("saved to", outfile)

Read it back to check (the header line starts with `#`, so `np.loadtxt` skips it automatically):

In [ ]:
with open(outfile) as f:
    print(f.read()[:300])

check = np.loadtxt(outfile)
print(np.allclose(check, table, rtol=1e-6))

(The `with open(...) as f:` block opens a file, lets you use it as `f`, and closes it automatically afterwards. `f.read()` returns the whole file as one string; `[:300]` shows the first 300 characters.)

In Colab, files you save live on a temporary virtual machine and disappear when the runtime disconnects. To keep a file, download it from the file browser (folder icon on the left) or save it to your Google Drive.

## Summary

- `import numpy as np`. An array holds many numbers of one `dtype` with a given `shape`.
- Create arrays: `np.array(list)`, `np.zeros(n)`, `np.ones(n)`, `np.full(n, value)`, `np.arange(start, stop, step)` (stop excluded), `np.linspace(start, stop, num)` (both ends included), `np.logspace`.
- Arithmetic (`+ - * / **`) and ufuncs (`np.sqrt`, `np.exp`, `np.sin`, `np.log`, ...) act element by element. This **vectorized** style is shorter and far faster than loops. Time code with `time.perf_counter()` or `%timeit`.
- Indexing `a[i]`, slicing `a[start:stop:step]`, 2D `M[row, col]`, `M[:, j]`. Slices are views: use `.copy()` for an independent copy.
- Boolean masks: `a[a > 0]`, combine with `&`, `|`, `~` and parentheses. `np.where(cond, x, y)` chooses element by element; `np.where(cond)` gives indices.
- Aggregates: `sum`, `mean`, `std` (`ddof=1` for sample std), `min`, `max`, `argmin`, `argmax`, `median`, with `axis=0` (down columns) or `axis=1` (across rows).
- `reshape`, `flatten`, `.T`; broadcasting stretches size-1 dimensions, e.g. column `[:, np.newaxis]` with a row gives a 2D table.
- Files: `np.loadtxt(path, unpack=True, delimiter=..., skiprows=...)`, `np.savetxt(path, np.column_stack([...]), header=..., fmt=...)`.

## Further reading

- NumPy: [the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) (official, free).
- J. VanderPlas, *Python Data Science Handbook*, Chapter 2 "Introduction to NumPy" ([free online](https://jakevdp.github.io/PythonDataScienceHandbook/)).
- M. Newman, *Computational Physics*, Section 2.4 (arrays); the chapters are partly available free on the author's website.